In [1]:
from sedona.spark import *
config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Setting Spark log level to "WARN".
26/03/02 19:29:37 INFO core/src/lib.rs: Sedona native acceleration engine v0.12.0 ready
                                                                                

In [2]:
database = 'gde_silver'

In [3]:
# Home Area Buffers

sedona.sql(f'''
create or replace table org_catalog.{database}.homes_buffers as
    select                                                                                                      
        sale_id,                                                                                                  
        ST_Buffer(geometry, 1600, true) as buffer_1mile
    from org_catalog.gde_bronze.king_co_homes 
''')

DataFrame[]

In [5]:
# Flood Hazards Table

sedona.sql(f'''
create or replace table org_catalog.{database}.homes_demographics as
  with overlaps as (
    select                                                                                                      
      a.sale_id,   
      try_cast(b.total_pop as double) as total_pop,
      try_cast(b.median_age as double) as median_age,
      try_cast(b.median_income as double) as median_income,
      st_area(
          st_intersection(a.buffer_1mile, b.geometry)
      ) / st_area(b.geometry) as share                                                                                             
    from org_catalog.{database}.homes_buffers a 
    join org_catalog.{database}.census_data b  
      on st_intersects(b.geometry, a.buffer_1mile)
  ),
  weighted AS (
    SELECT
      sale_id,
      (total_pop * share) AS pop_in_overlap,  
      median_age,
      median_income
    FROM overlaps
  )
  SELECT
    sale_id,
    SUM(pop_in_overlap) AS est_total_pop,
    SUM(median_income * pop_in_overlap) / NULLIF(SUM(pop_in_overlap), 0) AS pw_median_income_proxy,
    SUM(median_age * pop_in_overlap) / NULLIF(SUM(pop_in_overlap), 0) AS pw_median_age_proxy
  FROM weighted
  GROUP BY sale_id
''')

DataFrame[]

In [6]:
import wkls

washington = wkls.us.wa.wkt()
seattle = wkls.us.wa.seattle.wkt()
kirkland = wkls.us.wa.kirkland.wkt()
bellevue = wkls.us.wa.bellevue.wkt()

# Small bounding box in central Kirkland (~0.5km²)
kirkland_small = "POLYGON((-122.212 47.676, -122.200 47.676, -122.200 47.682, -122.212 47.682, -122.212 47.676))"

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [7]:
# Zoning Percentages

sedona.sql(f'''
create or replace table org_catalog.{database}.zoning_overlaps as
    select                                                                                                      
        a.sale_id,   
        b.MASTER_CAT,
        b.SUB_CAT,
        st_area(st_intersection(a.buffer_1mile, b.geometry)) / st_area(b.geometry) as share                                                                                             
    from org_catalog.{database}.homes_buffers a 
    join org_catalog.gde_bronze.gen_land_use_bronze b  
        on st_intersects(b.geometry, a.buffer_1mile)
    where 
        st_isvalid(geometry) is true AND
        ST_Intersects(b.geometry, ST_GeomFromWKT('{kirkland_small}'))
''')

DataFrame[]

In [8]:
sedona.sql(f'''
create or replace table org_catalog.{database}.homes_zoning_overlaps as
SELECT
  sale_id,
  SUM(CASE WHEN master_cat = 'Industrial' THEN share ELSE 0.0 END) AS industrial,
  SUM(CASE WHEN sub_cat = 'Residential (12+ Units/Acre)' THEN share ELSE 0.0 END) AS urban_residential,
  SUM(CASE WHEN sub_cat = 'Mixed Use' THEN share ELSE 0.0 END) AS mixed_use,
  SUM(CASE WHEN sub_cat = 'Commercial/Office' THEN share ELSE 0.0 END) AS commercial,
  SUM(CASE WHEN master_cat = 'Active Open Space and Recreation' THEN share ELSE 0.0 END) AS recreation,
  SUM(CASE WHEN master_cat = 'Urban Character Residential' THEN share ELSE 0.0 END) AS light_urban,
  SUM(CASE WHEN master_cat = 'Rural Character Residential' THEN share ELSE 0.0 END) AS rural
FROM org_catalog.{database}.zoning_overlaps
GROUP BY sale_id
''')

DataFrame[]

In [9]:
sedona.sql("SHOW TABLES IN org_catalog.gde_silver").show()

+----------+--------------------+-----------+
| namespace|           tableName|isTemporary|
+----------+--------------------+-----------+
|gde_silver|         census_data|      false|
|gde_silver|       homes_buffers|      false|
|gde_silver|  homes_demographics|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver|homes_distance_to...|      false|
|gde_silver| homes_flood_hazards|      false|
|gde_silver| homes_school_scores|      false|
|gde_silver|homes_seismic_haz...|      false|
|gde_silver|homes_zoning_over...|      false|
|gde_silver|house_sales_ndvi_...|      false|
|gde_silver|house_sales_tri_s...|      false|
|gde_silver|house_sales_zonal...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|residential_build...|      false|
|gde_silver|     roads_proximity|      false|
|gde_silver|         school_data|      false|
|gde_silver|     zoning_overlaps|      false|
+----------+--------------------+-----------+



26/03/02 19:39:25 ERROR Utils: Uncaught exception in thread kubernetes-executor-pod-polling-sync
io.fabric8.kubernetes.client.KubernetesClientException: Failure executing: GET at: https://kubernetes.default.svc.cluster.local:443/api/v1/namespaces/742x8tgnc6/pods?labelSelector=spark-app-selector%3Dspark-686ddf48fbfb42f7a983508d1f68d493%2Cspark-role%3Dexecutor%2Cspark-exec-inactive%21%3Dtrue. Message: Unauthorized. Received status: Status(apiVersion=v1, code=401, details=null, kind=Status, message=Unauthorized, metadata=ListMeta(_continue=null, remainingItemCount=null, resourceVersion=null, selfLink=null, additionalProperties={}), reason=Unauthorized, status=Failure, additionalProperties={}).
	at io.fabric8.kubernetes.client.KubernetesClientException.copyAsCause(KubernetesClientException.java:205)
	at io.fabric8.kubernetes.client.dsl.internal.OperationSupport.waitForResult(OperationSupport.java:507)
	at io.fabric8.kubernetes.client.dsl.internal.BaseOperation.list(BaseOperation.java:451)
